# Анализ временного ряда температуры воды

Итоговое задание по курсу анализа временных рядов. Данные: Brisbane Water Quality Monitoring Dataset. Целевой ряд: дневная средняя температура воды `Temperature`.

## 1. Импорт и подготовка окружения

Основная логика вынесена в `src/time_series_pipeline.py`, чтобы notebook был воспроизводимым и не дублировал весь код пайплайна.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT))

import pandas as pd

from src.time_series_pipeline import (
    RAW_PATH, PROCESSED_PATH, METRICS_PATH, ANOMALIES_PATH, SUMMARY_PATH,
    BACKTESTING_PATH, PREDICTION_INTERVALS_PATH, RELIABILITY_PATH,
    TARGET, load_and_prepare, run_pipeline
)

## 2. Загрузка и первичный анализ данных

In [ ]:
raw = pd.read_csv(RAW_PATH, parse_dates=["Timestamp"])
print(raw.shape)
raw.head()

In [ ]:
raw.info()

In [ ]:
missing = raw.isna().sum().sort_values(ascending=False)
missing[missing > 0]

## 3. Подготовка дневного ряда

Повторяющиеся временные метки усредняются, затем ряд ресемплируется до дневной частоты. Пропуски заполняются временной интерполяцией.

In [ ]:
daily = load_and_prepare()
print(daily.shape)
daily[[TARGET, "pH", "Turbidity", "Dissolved Oxygen", "Salinity"]].describe()

In [ ]:
daily[[TARGET]].plot(figsize=(12, 4), title="Daily water temperature")

## 4. Запуск пайплайна

Пайплайн обучает baseline, статистические, ML и DL модели, считает метрики и строит графики.

In [ ]:
summary = run_pipeline()
summary

## 5. Сравнение моделей

In [ ]:
metrics = pd.read_csv(METRICS_PATH)
metrics

In [ ]:
metrics.sort_values("MASE").head(5)

## 6. Аномалии

In [ ]:
pd.read_csv(ANOMALIES_PATH)

## 7. Графики отчета

![Daily water temperature](../reports/figures/temperature_series.png)

![STL decomposition](../reports/figures/stl_decomposition.png)

![Forecast comparison](../reports/figures/forecast_comparison.png)

![Rolling-origin backtesting](../reports/figures/backtesting_mase.png)

![Prediction intervals](../reports/figures/prediction_intervals.png)

![Anomalies](../reports/figures/anomalies.png)

## 8. Backtesting и интервальные оценки

Этот блок соответствует требованию проверки надежности: rolling-origin backtesting на 4 окнах по 14 дней и эмпирические интервалы прогноза для лучшей модели.

Rolling-origin таблица показывает устойчивость моделей на нескольких тестовых окнах, а интервалы прогноза показывают фактическое покрытие лучшей holdout-модели.

In [ ]:
backtesting = pd.read_csv(BACKTESTING_PATH)
backtesting

In [ ]:
intervals = pd.read_csv(PREDICTION_INTERVALS_PATH)
print(RELIABILITY_PATH.read_text(encoding="utf-8"))
intervals.head()

## 9. Вывод

На 60-дневном holdout лучшей моделью стала Ridge regression with lag features: MAE = 1.925, RMSE = 2.468, sMAPE = 9.49%, MASE = 1.482. Rolling-origin backtesting показывает, что на коротких 14-дневных окнах наиболее устойчив Holt linear trend с MASE_mean = 0.726, поэтому итоговая рекомендация: использовать Ridge как основную holdout-модель, а Holt как статистический контроль. Подробные выводы по задачам №1-№4 приведены в README.